In [3]:
pip install langchain_google_genai langchain_tavily langchain_classic

  Using cached langchain_google_genai-4.4.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_tavily-0.2.18-py3-none-any.whl.metadata (20 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 10.2 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are insta

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from langchain_classic.agents import tool
import smtplib
from langchain.agents import create_agent
import requests
import os
from google.colab import userdata

In [10]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=userdata.get("google_api_key")
)

In [24]:
@tool
def web_search(query):
  "search web about something by giving it the topic of search"
  search=TavilySearch(
    tavily_api_key=userdata.get("TAVILY_API_KEY"),
    max_results=5 , #number of sites to search
    topic="general",
    search_depth="advanced",
)
  results=search.invoke(query)
  return results

@tool
def send_email(receiver_email,subject,message):
  "this tool can be used to send email to person by giving receiver email , subject , message"
  sender_email="muhamedabdelraheem22@gmail.com"
  text=f"subject :{subject} \n\n{message}"
  server=smtplib.SMTP("smtp.gmail.com" , 587)
  server.starttls()
  server.login(sender_email,userdata.get('sender_passkey'))
  server.sendmail(sender_email,receiver_email,text)

@tool
def send_to_telegram(msg):
  "this is used when need to send a message via telegram"
  user_id="5089266163"
  bot_token=userdata.get('bot_token')
  url=f"https://api.telegram.org/bot{bot_token}/sendmessage?chat_id={user_id}&text={msg}"
  requests.get(url)
  return "message has been successfully sent"

In [25]:
tools=[web_search,send_email,send_to_telegram]

In [26]:
agent=create_agent(
    model=llm,
    tools=tools,
    system_prompt="you are an helpful assistant"
)

In [27]:
result=agent.invoke(
    {"messages" : [{"role" : "user" , "content" : "i want you to summarize article about claude opus 5.5 and send it to my mail'om9556or@gmail.com' in english and send it telegram in arabic"}]}
)

In [28]:
result

{'messages': [HumanMessage(content="i want you to summarize article about claude opus 5.5 and send it to my mail'om9556or@gmail.com' in english and send it telegram in arabic", additional_kwargs={}, response_metadata={}, id='da8c34cc-d453-46b6-bb85-31372915be6f'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "Claude Opus 3.5 update news (checking if 5.5 exists)"}'}, '__gemini_function_call_thought_signatures__': {'call_52331': 'EnMKcQFpFH0TgRqa+kFPhUM5NtPPyn22nvEvaZiywTUT2T1ig24mH8CCg3HVOEw6Pnl41tlfZK2QQIPuSGcIHVe2vjqijdzQt48F28ByoHj43HKQo77AW62yM9cElO5vxDkX9vW24pXhq5FzGYZI+sWS2Q76'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0ed05-9e35-7f82-a472-c811cb1ff561-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'Claude Opus 3.5 update news (checking if 5.5 exists)'}, 'id': 'call_52331', 'type': 'tool_call'}],